In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import gc

# ==========================================
# 0. CONFIGURATION & DATA LOADING
# ==========================================
train_path = "../Datasets/ts-forecasting/train.parquet"
TARGET = "y_target"
FORECAST_WINDOWS = [1, 3, 10, 25]
RESULTS_CSV = "random_search_results.csv"

# Initialize our tracking CSV with headers
header = pd.DataFrame(columns=[
    'horizon', 'iteration', 'train_wrmse', 'holdout_wrmse', 
    'learning_rate', 'n_estimators', 'num_leaves', 'min_child_samples', 
    'feature_fraction', 'bagging_fraction', 'bagging_freq'
])
header.to_csv(RESULTS_CSV, mode='a',index=False)

# ==========================================
# 1. METRICS & TRUE RANDOM SAMPLER
# ==========================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def get_random_params():
    """
    True Random Search: Samples from continuous distributions and full integer ranges.
    No discrete grids.
    """
    return {
        'objective': 'regression',
        'metric': 'rmse', # LightGBM's internal stopping will use standard RMSE
        # Log-uniform distribution for learning rate (e.g., between 0.001 and 0.1)
        'learning_rate': float(np.exp(np.random.uniform(np.log(0.001), np.log(0.1)))),
        # Full integer ranges
        'n_estimators': int(np.random.randint(500, 4000)),
        'num_leaves': int(np.random.randint(15, 256)),
        'min_child_samples': int(np.random.randint(10, 300)),
        'bagging_freq': int(np.random.randint(1, 15)),
        # Continuous uniform distributions
        'feature_fraction': float(np.random.uniform(0.4, 1.0)),
        'bagging_fraction': float(np.random.uniform(0.4, 1.0)),
        
        'verbosity': -1,
        'n_jobs': -1,
        'random_state': 42 # Ensures tree-building reproducibility within the run
    }

# ==========================================
# 2. DATA PREP & CACHING (Saves time in loop)
# ==========================================
print("Loading train.parquet and applying consistent 80/20 splits...")
train_full = pd.read_parquet(train_path)

horizon_cache = {}

for horizon in FORECAST_WINDOWS:
    tr_df = train_full[train_full['horizon'] == horizon].reset_index(drop=True)

    if len(tr_df) == 0:
        continue

    # Clean features
    drop_cols = [TARGET, 'ts_index', 'horizon', 'id']
    features = sorted([c for c in tr_df.columns if c not in drop_cols])

    X = tr_df[features].copy()
    
    # Cast to numpy immediately to prevent index mismatch issues in custom metric
    y = tr_df[TARGET].values
    weights = tr_df['weight'].values if 'weight' in tr_df.columns else np.ones_like(y)

    # Cast categoricals globally before splitting
    cat_cols = X.select_dtypes(exclude=['number', 'bool']).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype('category')

    # Fixed 80/20 Holdout Split
    X_train, X_valid, y_train, y_valid, w_train, w_valid = train_test_split(
        X, y, weights, test_size=0.20, random_state=42, shuffle=True
    )
    
    horizon_cache[horizon] = (X_train, X_valid, y_train, y_valid, w_train, w_valid)
    print(f"✅ Horizon {horizon} prepped: Train={len(X_train)} rows | Holdout={len(X_valid)} rows")

# Free up memory now that we have our exact arrays cached
del train_full
gc.collect()

# ==========================================
# 3. CONTINUOUS RANDOM SEARCH LOOP
# ==========================================
print("\n🚀 Starting continuous round-robin search. Kill the script manually when satisfied.")

iteration = 1
best_scores = {h: -1.0 for h in horizon_cache.keys()}

while True:
    print(f"\n{'='*50}\n 🔄 GLOBAL ITERATION: {iteration}\n{'='*50}")
    
    for horizon, (X_train, X_valid, y_train, y_valid, w_train, w_valid) in horizon_cache.items():
        params = get_random_params()
        
        # Train Model
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            sample_weight=w_train,
            eval_set=[(X_valid, y_valid)],
            eval_sample_weight=[w_valid],
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )
        
        # Predict
        pred_train = model.predict(X_train)
        pred_valid = model.predict(X_valid)
        
        # Score using custom metric (Higher is Better)
        score_train = weighted_rmse_score(y_train, pred_train, w_train)
        score_valid = weighted_rmse_score(y_valid, pred_valid, w_valid)
        
        # Track Best Model
        is_best = ""
        if score_valid > best_scores[horizon]:
            best_scores[horizon] = score_valid
            is_best = "🌟 NEW BEST!"
            
        print(f"Hz {horizon:<2} | Train WRMSE: {score_train:.5f} | Holdout WRMSE: {score_valid:.5f} | Trees: {model.best_iteration_:<4} {is_best}")
        
        # Append immediately to CSV so no data is lost
        run_results = pd.DataFrame([{
            'horizon': horizon,
            'iteration': iteration,
            'train_wrmse': score_train,
            'holdout_wrmse': score_valid,
            'learning_rate': params['learning_rate'],
            'n_estimators': model.best_iteration_, # Logs exact trees before early stopping
            'num_leaves': params['num_leaves'],
            'min_child_samples': params['min_child_samples'],
            'feature_fraction': params['feature_fraction'],
            'bagging_fraction': params['bagging_fraction'],
            'bagging_freq': params['bagging_freq']
        }])
        run_results.to_csv(RESULTS_CSV, mode='a', header=False, index=False)
        
    iteration += 1

Loading train.parquet and applying consistent 80/20 splits...
✅ Horizon 1 prepped: Train=1115722 rows | Holdout=278931 rows
✅ Horizon 3 prepped: Train=1108652 rows | Holdout=277164 rows
✅ Horizon 10 prepped: Train=1069788 rows | Holdout=267448 rows
✅ Horizon 25 prepped: Train=975767 rows | Holdout=243942 rows

🚀 Starting continuous round-robin search. Kill the script manually when satisfied.

 🔄 GLOBAL ITERATION: 1


KeyboardInterrupt: 